# notebooks/05a_gnn_architecture.ipynb — Markdown Cell

# Phase 5A — Deterministic GNN Architecture

## Phase Status

This notebook analyzes deterministic GNN models trained in Phase 5A.

Phase 5A is the first part of the main research contribution. It evaluates whether graph-based message passing improves wildfire burn probability prediction compared with the strongest non-graph baselines from Phase 4 and Phase 4B.

---

## 1. Scientific Motivation

Previous phases established strong baselines:

- Random Forest and XGBoost use tabular node features.
- CNN uses local raster patches and convolutional spatial context.
- GNN uses explicit graph topology and message passing.

The key scientific question in Phase 5A is:

> Does graph topology add predictive value beyond tabular features and CNN spatial convolution?

This is important because the strongest baseline is now the CNN spatial model.

```text
Best Phase 4 model: XGBoost, R² = 0.6761
Best Phase 4B model: CNN, R² = 0.7187

GCN

>python scripts/phase5a_run_gnn.py --model gcn --epochs 50 --device cpu


===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GCNModel(
  (convs): ModuleList(
    (0): GCNConv(61, 64)
    (1-2): 2 x GCNConv(64, 64)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=5614.34375 | val_loss=13449.08105 | lr=1.00e-03
Epoch 005/50 | train_loss=527.05066 | val_loss=903.23206 | lr=1.00e-03
Epoch 010/50 | train_loss=155.96542 | val_loss=3.45268 | lr=1.00e-03
Epoch 015/50 | train_loss=46.73463 | val_loss=3.61016 | lr=1.00e-03
Epoch 020/50 | train_loss=13.01342 | val_loss=0.79873 | lr=1.00e-03
Epoch 025/50 | train_loss=3.91892 | val_loss=1.55369 | lr=1.00e-03
Epoch 030/50 | train_loss=1.51573 | val_loss=1.36348 | lr=5.00e-04
Early stopping at epoch 34

Training finished in 227.8s
Best val_loss: 0.73921

Predicting on test split...

  ── GCN ──
    R²       =  -0.0179   (>0 = beats naive mean predictor)
    MAE      =  0.02182  (burn prob scale ~[0, 0.25])
    Spearman =   0.6773   (rank correlation, robust)
    Brier    =  0.00142  (lower = better calibration)
    ECE      =  0.00936  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506 -152.023    0.00917      0.083
    2      [0.0040, 0.0099]    11,505 -122.540    0.01018      0.282
    3      [0.0099, 0.0205]    11,507 -107.139    0.01538      0.239
    4      [0.0205, 0.0465]    11,506  -26.181    0.01975      0.043
    5      [0.0465, 0.2082]    11,507   -2.007    0.05463      0.390 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_gcn_test_predictions.npz
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_gcn_best.pt

Next run:
python scripts/phase5a_make_figures.py


Sage:

 scripts/phase5a_run_gnn.py --model graphsage --epochs 50 --device cpu

===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GraphSAGEModel(
  (convs): ModuleList(
    (0): SAGEConv(61, 64, aggr=mean)
    (1-2): 2 x SAGEConv(64, 64, aggr=mean)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=482.09665 | val_loss=1105.94177 | lr=1.00e-03
Epoch 005/50 | train_loss=85.65029 | val_loss=7.82571 | lr=1.00e-03
Epoch 010/50 | train_loss=11.73873 | val_loss=0.72938 | lr=1.00e-03
Epoch 015/50 | train_loss=2.42194 | val_loss=1.44303 | lr=1.00e-03
Epoch 020/50 | train_loss=1.15096 | val_loss=1.49840 | lr=5.00e-04
Early stopping at epoch 22

Training finished in 83.6s
Best val_loss: 0.72938

Predicting on test split...

  ── GRAPHSAGE ──
    R²       =   0.4424   (>0 = beats naive mean predictor)
    MAE      =  0.01688  (burn prob scale ~[0, 0.25])
    Spearman =   0.7468   (rank correlation, robust)
    Brier    =  0.00078  (lower = better calibration)
    ECE      =  0.01066  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506  -81.527    0.00943     -0.094
    2      [0.0040, 0.0099]    11,505  -15.829    0.00610      0.191
    3      [0.0099, 0.0205]    11,507   -6.176    0.00440      0.154
    4      [0.0205, 0.0465]    11,506   -6.587    0.01409      0.314
    5      [0.0465, 0.2082]    11,507   -1.379    0.05037      0.625 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_graphsage_test_predictions.npz       
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_graphsage_best.pt

Next run:
python scripts/phase5a_make_figures.py


GAT:

_gnn>python scripts/phase5a_run_gnn.py --model gat --epochs 50 --device cpu --hidden-channels 16 --heads 2 --num-layers 2 --dropout 0.3

===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GATModel(
  (convs): ModuleList(
    (0): GATConv(61, 16, heads=2)
    (1): GATConv(32, 16, heads=2)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=32, out_features=16, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=16, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=9362.38965 | val_loss=2052.49585 | lr=1.00e-03
Epoch 005/50 | train_loss=5050.90967 | val_loss=56.62851 | lr=1.00e-03
Epoch 010/50 | train_loss=2637.91528 | val_loss=576.14026 | lr=1.00e-03
Epoch 015/50 | train_loss=1583.20667 | val_loss=985.64069 | lr=5.00e-04
Early stopping at epoch 17

Training finished in 69.4s
Best val_loss: 56.62851

Predicting on test split...

  ── GAT ──
    R²       = -15.4094   (>0 = beats naive mean predictor)
    MAE      =  0.12658  (burn prob scale ~[0, 0.25])
    Spearman =   0.3540   (rank correlation, robust)
    Brier    =  0.02290  (lower = better calibration)
    ECE      =  0.12504  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506 -13925.659    0.09539      0.219
    2      [0.0040, 0.0099]    11,505 -7265.624    0.11686      0.083
    3      [0.0099, 0.0205]    11,507 -3214.022    0.14808      0.148
    4      [0.0205, 0.0465]    11,506 -579.860    0.15361     -0.001
    5      [0.0465, 0.2082]    11,507  -11.052    0.11898      0.226 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_gat_test_predictions.npz
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_gat_best.pt

Next run:
python scripts/phase5a_make_figures.py
